# 07 - Calibrated Abstention via Split-Conformal Prediction (item 3)

Layers split-conformal prediction (LAC score) on top of the learned verifier from notebook 05. Runs **two** calibration strategies, reported side by side -- see `calibration.py`'s module docstring for why both matter:

1. **Silver-calibrated, gold-evaluated**: calibrate the abstention threshold on a large held-out slice of silver (rule-based) labels, then check whether the nominal coverage (e.g. 90%) actually holds on the human-adjudicated gold set. This is *not* a valid conformal guarantee in the textbook sense (silver and gold aren't exchangeable) -- the gap between nominal and achieved coverage IS the finding.
2. **Gold-internal split**: calibrate and test entirely within gold (50/50 split). This one is methodologically valid, but at ~70 usable gold claims the calib/test halves are only ~35 each, so read the coverage estimate together with its bootstrap CI, not just the point value.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import json
from graph_utils import CitationGraph, load_relational_claims
from hard_negatives import build_tfidf_index
from verifier_features import build_dataset
import verifier_model as vm
from calibration import run_calibration_suite, split_indices
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

graph = CitationGraph.load(REFS_CACHE)
all_claims = load_relational_claims(BENCH, CITE_RESULTS)

paper_abstracts = {}
with open(BENCH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for gt in rec.get("ground_truth_papers") or []:
            pid, ab = gt.get("paper_id"), gt.get("abstract")
            if pid and ab:
                paper_abstracts[pid] = ab
tfidf_vectors = build_tfidf_index(paper_abstracts)

silver = vm.load_silver_labels(CITE_RESULTS)
gold = vm.load_gold_labels(BENCH_VERIFIED) if BENCH_VERIFIED else {}
silver_clean = {k: v for k, v in silver.items() if k not in gold}

X_silver, y_silver, meta_silver = build_dataset(all_claims, graph, tfidf_vectors, silver_clean)
X_gold, y_gold, meta_gold = build_dataset(all_claims, graph, tfidf_vectors, gold)
print(f"Silver: {X_silver.shape} | Gold: {X_gold.shape}")

train_idx, calib_idx = split_indices(len(y_silver), frac=0.85, seed=0)
X_tr, y_tr = X_silver[train_idx], y_silver[train_idx]
X_cal, y_cal = X_silver[calib_idx], y_silver[calib_idx]
print(f"Silver train: {len(y_tr)} | Silver calibration slice: {len(y_cal)}")

scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(scaler.transform(X_tr), y_tr)

results = run_calibration_suite(model, scaler, X_cal, y_cal, X_gold, y_gold, alphas=(0.1, 0.2), seed=0)

for alpha_key, block in results["alphas"].items():
    print(f"\n--- {alpha_key} (target coverage {block['target_coverage']:.0%}) ---")
    s = block["silver_calibrated_on_gold"]
    print(f"  silver-calibrated -> gold: coverage={s['coverage']:.3f} "
          f"[{s['coverage_ci'][0]:.3f},{s['coverage_ci'][1]:.3f}] "
          f"abstention_rate={s['abstention_rate']:.3f} avg_set_size={s['avg_set_size']:.3f} "
          f"selective_acc={s['selective_accuracy']}")
    g = block["gold_internal_split"]
    if "coverage" in g:
        print(f"  gold-internal split:      coverage={g['coverage']:.3f} "
              f"[{g['coverage_ci'][0]:.3f},{g['coverage_ci'][1]:.3f}] "
              f"abstention_rate={g['abstention_rate']:.3f} avg_set_size={g['avg_set_size']:.3f} "
              f"(n_calib={g['n_calib']}, n_test={g['n']})")
    else:
        print("  gold-internal split:", g.get("note"))

with open(OUT_DIR / "calibration_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("\nSaved to", OUT_DIR / "calibration_results.json")
